# Experiment: Ryzen 9700X V/F Analysis

This notebook uses the reusable `vfanalysis` package to run a clean pipeline:
`load_data -> add_all_features -> filter_df -> metrics -> plots`.

Interpretations are exploratory and should not be treated as proof of per-core Curve Optimizer stability.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd

from vfanalysis.features import add_all_features
from vfanalysis.filters import base_filter_config, filter_df, vf_filter_config
from vfanalysis.io import load_processed_dataset
from vfanalysis.metrics import boost_ridge, core_summary
from vfanalysis.plots import (
    plot_boost_ridge,
    plot_clock_variance_by_power,
    plot_power_clock_hexbin,
    plot_vf_curves,
    plot_vf_hexbin_per_core,
)
from vfanalysis.regressions import (
    fit_linear_regression,
    fit_power_law_regression,
    fit_sklearn_exploratory_regressions,
    per_core_fit_summary,
    thermal_exploratory_regressions,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 1) Load Processed Dataset

Loads all parquet files from `VF_DATASET_DIR`.


In [ ]:
df_raw = load_processed_dataset()

print(f"Rows: {len(df_raw):,}")
print(f"Cores: {sorted(df_raw['core'].dropna().unique().tolist())}")
print(f"Runs: {df_raw['run_id'].nunique()}")

df_raw.head()


## 2) Feature Engineering + Reusable Filters

`add_all_features` is deterministic and row-wise only.


In [ ]:
df_features = add_all_features(df_raw)

df_base = filter_df(df_features, base_filter_config())
df_vf = filter_df(df_base, vf_filter_config())

print(f"Raw rows:      {len(df_raw):,}")
print(f"Base filtered: {len(df_base):,}")
print(f"VF filtered:   {len(df_vf):,}")


## 3) Per-Core V/F/P/T Visualizations


In [ ]:
plot_vf_hexbin_per_core(df_vf, freq_col="eff_clock")
plot_vf_hexbin_per_core(df_vf, freq_col="clock")
plot_power_clock_hexbin(df_base)
plt.show()


In [ ]:
plot_vf_curves(df_vf, bins=40, quantile=0.99)
plot_boost_ridge(df_base, power_min=10.0)
plot_clock_variance_by_power(df_base, power_min=1.0, clock_min=3000)
plt.show()


## 4) Metrics


In [ ]:
summary = core_summary(df_vf).sort_values("core")
summary


In [ ]:
ridge_df = boost_ridge(df_base, power_min=10.0)
ridge_df.head(20)


## 5) Regression Diagnostics (Exploratory)

These fits are descriptive/exploratory, not causal proof.


In [ ]:
linear_per_core = per_core_fit_summary(df_vf, fit_fn=fit_linear_regression)
power_law_per_core = per_core_fit_summary(df_vf, fit_fn=fit_power_law_regression)

print("Linear regression per core:")
linear_per_core[["core", "r2", "residual_std", "coefficients"]]


In [ ]:
print("Power-law regression per core:")
power_law_per_core[["core", "r2", "residual_std", "coefficients"]]


In [ ]:
# Notebook-style sklearn exploratory regressions (linear + multivariate)
reg_rows = []
for core in sorted(df_vf["core"].dropna().unique()):
    reg = fit_sklearn_exploratory_regressions(df_vf, core=int(core))
    reg_rows.append(reg)

regression_summary = pd.concat(reg_rows, ignore_index=True)
regression_summary


In [ ]:
thermal_summary = thermal_exploratory_regressions(df_base)
thermal_summary[["spec", "r2", "residual_std", "n_samples"]].sort_values("r2", ascending=False)


## 6) Observation Notes (With Uncertainty)

Use this section to document what is supported by data and what remains uncertain.


In [ ]:
max_eff_clock = summary.set_index("core")["max_eff_clock"].round(1).to_dict()
median_clock_per_power = summary.set_index("core")["clock_per_power"].round(2).to_dict()

observations = {
    "max_eff_clock_by_core_mhz": max_eff_clock,
    "median_clock_per_power_by_core": median_clock_per_power,
    "note": (
        "Per-core CO implications are exploratory estimates only. "
        "Use dedicated stability testing before treating inferred ranking as actionable."
    ),
}

observations
